# Custom Task Evaluation: Few-Shot Sensitivity and Task-Specific Metrics

General benchmarks tell you little about whether a model will work for your specific application. Custom task evaluation — designing prompts, collecting examples, and measuring the right metrics for your use case — is the practical evaluation skill that matters most in production.

## Why General Benchmarks Are Insufficient

A model scoring 85% on MMLU may perform at 60% on your specific task if:
- Your domain is underrepresented in the benchmark
- The task requires multi-step reasoning not tested in standard benchmarks
- Your prompt format differs from the few-shot examples the model was tuned on
- Your evaluation metric (e.g. format compliance) is not part of any standard benchmark

Building a custom evaluation set is the only way to get reliable estimates of production performance.

## Few-Shot Sensitivity: A Critical Stability Test

LLM performance is highly sensitive to the choice of few-shot examples. A task that achieves 80% accuracy with one random seed's examples might get only 65% with another set. This variance is often larger than the difference between models.

Before comparing models on a custom task, you must characterize and control for this variance by:
1. Running multiple random seeds
2. Reporting mean and standard deviation, not just peak performance
3. Using stratified sampling to ensure few-shot examples are representative

In [ ]:
from dataclasses import dataclass
import random
import statistics

@dataclass
class SensitivityResult:
    mean: float
    std: float
    min_acc: float
    max_acc: float
    n_seeds: int

def run_sensitivity(eval_set: list, pool: list, model_fn, shots: int = 5,
                    n_seeds: int = 10) -> SensitivityResult:
    accs = []
    for seed in range(n_seeds):
        rng = random.Random(seed)
        few_shot_examples = rng.sample(pool, min(shots, len(pool)))
        prefix = '\n'.join(f'Input: {e["input"]}\nOutput: {e["output"]}' for e in few_shot_examples)
        correct = 0
        for example in eval_set:
            prompt = prefix + f'\nInput: {example["input"]}\nOutput:'
            pred = model_fn(prompt).strip()
            if pred.lower() == example['output'].lower():
                correct += 1
        accs.append(correct / len(eval_set))
    return SensitivityResult(
        mean=statistics.mean(accs),
        std=statistics.stdev(accs) if len(accs) > 1 else 0.0,
        min_acc=min(accs),
        max_acc=max(accs),
        n_seeds=n_seeds
    )

# Simulated sentiment classification task
train_pool = [
    {'input': 'The product exceeded my expectations', 'output': 'positive'},
    {'input': 'Terrible quality, broke in a week', 'output': 'negative'},
    {'input': 'It works as advertised', 'output': 'neutral'},
    {'input': 'Absolutely love this item', 'output': 'positive'},
    {'input': 'Would not recommend to anyone', 'output': 'negative'},
    {'input': 'Decent but nothing special', 'output': 'neutral'},
    {'input': 'Best purchase I have made this year', 'output': 'positive'},
    {'input': 'Stopped working after one month', 'output': 'negative'},
]
test_set = [
    {'input': 'Really happy with this', 'output': 'positive'},
    {'input': 'Complete waste of money', 'output': 'negative'},
    {'input': 'Does what it says', 'output': 'neutral'},
]

# Mock model with slight bias toward positive
def biased_model(prompt):
    if 'love' in prompt or 'excellent' in prompt or 'happy' in prompt:
        return 'positive'
    if 'terrible' in prompt or 'broken' in prompt or 'waste' in prompt:
        return 'negative'
    return 'positive'  # default bias

result = run_sensitivity(test_set, train_pool, biased_model, shots=3, n_seeds=8)
print(f'Mean accuracy: {result.mean:.1%}')
print(f'Std deviation: {result.std:.1%}')
print(f'Range: [{result.min_acc:.1%}, {result.max_acc:.1%}]')
print(f'Note: range of {result.max_acc - result.min_acc:.1%} shows few-shot sensitivity')

## Designing Task-Specific Metrics

For production tasks, accuracy against a label is often not the right metric:

- **Format compliance**: does the output follow the required JSON schema, character limit, or tone?
- **Entity precision/recall**: for extraction tasks, are the right entities extracted without hallucination?
- **Semantic equivalence**: is the output meaning-equivalent to the reference, even if worded differently?
- **Safety pass rate**: does the model refuse when it should and respond when it should?

Define your metric before collecting examples, and design your evaluation set to stress-test the specific failure modes you care about.

In [ ]:
def evaluate_format_compliance(outputs: list, schema: dict) -> dict:
    import json as _json
    required_keys = schema.get('required', [])
    results = []
    for out in outputs:
        try:
            parsed = _json.loads(out)
            has_all_keys = all(k in parsed for k in required_keys)
            results.append({'valid_json': True, 'has_required_keys': has_all_keys,
                            'compliant': has_all_keys})
        except _json.JSONDecodeError:
            results.append({'valid_json': False, 'has_required_keys': False, 'compliant': False})
    compliance_rate = sum(r['compliant'] for r in results) / len(results)
    json_rate = sum(r['valid_json'] for r in results) / len(results)
    return {'compliance_rate': compliance_rate, 'valid_json_rate': json_rate, 'n': len(results)}

schema = {'required': ['category', 'sentiment', 'confidence']}
outputs = [
    '{"category": "electronics", "sentiment": "positive", "confidence": 0.9}',
    '{"category": "books", "sentiment": "negative"}',  # missing confidence
    'The sentiment is positive.',  # not JSON
    '{"category": "clothing", "sentiment": "neutral", "confidence": 0.7}',
]
result = evaluate_format_compliance(outputs, schema)
print(f'Compliance rate: {result["compliance_rate"]:.0%}')
print(f'Valid JSON rate: {result["valid_json_rate"]:.0%}')

## The Evaluation Data Collection Problem

Building a reliable custom evaluation set requires:
1. **Coverage**: examples across difficulty levels, edge cases, and distribution shifts
2. **Label quality**: inter-annotator agreement; disagreement reveals ambiguity in your task definition
3. **Size**: enough examples for statistical significance — typically 200-500 for a first pass
4. **Separation from training**: evaluation examples must never appear in fine-tuning data

A common mistake is collecting evaluation data from the same distribution as training data. Always collect a test set from a slightly different source or time period to estimate generalization.